In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import re
from datetime import datetime

In [14]:
def cargar_sms_xml(ruta_xml: str) -> pd.DataFrame:
    tree = ET.parse(ruta_xml)
    root = tree.getroot()

    registros = []
    for sms in root.findall("sms"):
        registros.append({
            "address": sms.get("address"),
            "date_raw": sms.get("date"),
            "date": datetime.fromtimestamp(int(sms.get("date")) / 1000),
            "body": sms.get("body")
        })

    return pd.DataFrame(registros)

def extraer_local_y_valor(texto: str):
    """
    Extrae local y valor desde un SMS.
    Devuelve (local, valor_float) o (None, None) si no coincide.
    """
    patron = re.compile(
        r"por\s+(\d{1,3}(?:\.\d{3})*,\d{2})\s*,?\s*en\s+([^,]+)",
        re.IGNORECASE
    )

    m = patron.search(texto)
    if not m:
        return None, None

    local = m.group(2)
    valor = m.group(1)

    # Normalizar valor
    valor = valor.replace(".", "").replace(",", ".")
    valor = float(valor)

    return local, valor

def procesar_sms(df: pd.DataFrame) -> pd.DataFrame:
    df["local"] = None
    df["valor"] = None

    for i, body in df["body"].items():
        local, valor = extraer_local_y_valor(body)
        df.at[i, "local"] = local
        df.at[i, "valor"] = valor
    return df

def clasificar_sms(body: str) -> str:
    texto = body.lower()
    
    # Compra en comercio
    if re.search(r"pedidosya", texto):
        return "compra"

    # Pago (servicios, tarjetas, comercios)
    if re.search(r"pago|pagad[ao]|cancelación|cancelado", texto):
        return "pago"

    # Débito automático
    if re.search(r"débito|debito|cargo automático|cargo mensual", texto):
        return "debito_automatico"

    # Impuestos / retenciones
    if re.search(r"ret iva|retención|impuesto|salida divisas|isd", texto):
        return "impuesto_retencion"

    # Transferencias enviadas
    if re.search(r"transferencia enviada|transferido|enviado a", texto):
        return "transferencia_enviada"

    # Transferencias recibidas
    if re.search(r"transferencia recibida|recibido de|depositado", texto):
        return "transferencia_recibida"

    # Retiros
    if re.search(r"retiro|cajero|atm", texto):
        return "retiro"

    # Depósitos
    if re.search(r"depósito|deposito|acreditado", texto):
        return "deposito"

    # Si no coincide con nada
    return "otros"

def agregar_clasificacion(df):
    df["categoria"] = df["body"].apply(clasificar_sms)
    return df

In [15]:
df_sms = cargar_sms_xml("data/sms_backup.xml")   # tu pipeline actual
df_sms = procesar_sms(df_sms)          # extrae local + valor
df_sms = agregar_clasificacion(df_sms)    # agrega categoría
df_sms.to_csv("output/sms_procesados.csv", index=False, sep=";", decimal=",")
print(df_sms.head())


  address       date_raw                    date  \
0   +2626  1765288379512 2025-12-09 08:52:59.512   
1   +2626  1765288405008 2025-12-09 08:53:25.008   
2   +2626  1765288506670 2025-12-09 08:55:06.670   
3   +6007  1765313586501 2025-12-09 15:53:06.501   
4   +2667  1765392984067 2025-12-10 13:56:24.067   

                                                body local valor categoria  
0  BdP informa: Por tu seguridad NO compartas est...  None  None     otros  
1  Diego Vaca, ha registrado un nuevo equipo para...  None  None     otros  
2  BdP informa: Por tu seguridad NO compartas est...  None  None     otros  
3  Notificaci n sobre tu cuenta: la contrase a de...  None  None     otros  
4  EEQ Recomienda: Usa temporizadores para tus lu...  None  None     otros  
